In [1]:
import shutil
import tempfile
import time
from pathlib import Path

import numpy as np
import xarray as xr
import zarr

# Create a fresh temp directory for all zarr stores in this run
tmpdir = Path(tempfile.mkdtemp(prefix="zarr_explore_"))
print(f"Writing to: {tmpdir}")

n1 = 1000
n2 = 200

fname = str(tmpdir / "baseline.zarr")

t_start = time.perf_counter()
for i in range(n2):
    x1 = np.arange(n1)
    x2 = np.repeat([float(i)], n1)
    y1 = x1 * x2

    ds = xr.Dataset(
        {
            "y1": (["t"], y1),
        },
        coords={
            "x1": ("t", x1),
            "x2": ("t", x2),
        },
    )
    if i == 0:
        ds.to_zarr(
            fname,
            consolidated=False,
            encoding={
                "x1": {"chunks": (10000,)},
                "x2": {"chunks": (10000,)},
                "y1": {"chunks": (10000,)},
            },
        )
    else:
        ds.to_zarr(fname, append_dim="t", consolidated=False)
duration = time.perf_counter() - t_start

print(f"Baseline (xarray append_dim per iteration): {duration:.3f}s")

Writing to: C:\Users\jenielse\AppData\Local\Temp\zarr_explore_qpzbwqny
Baseline (xarray append_dim per iteration): 14.282s


## Strategy 1: Direct Zarr API (skip xarray overhead on append)

xarray's `to_zarr` with `append_dim` re-validates metadata each call. Using the zarr API directly to append raw numpy arrays avoids this overhead.

In [2]:
fname1 = str(tmpdir / "direct_zarr.zarr")

t_start = time.perf_counter()

# Create the zarr group with xarray for the first write (sets up metadata/attrs)
x1 = np.arange(n1)
x2 = np.repeat([0.0], n1)
y1 = x1 * x2
ds = xr.Dataset(
    {"y1": (["t"], y1)},
    coords={"x1": ("t", x1), "x2": ("t", x2)},
)
ds.to_zarr(
    fname1,
    consolidated=False,
    encoding={
        "x1": {"chunks": (10000,)},
        "x2": {"chunks": (10000,)},
        "y1": {"chunks": (10000,)},
    },
)

# Now open with zarr directly and append
root = zarr.open(fname1, mode="r+")
for i in range(1, n2):
    x1 = np.arange(n1)
    x2 = np.repeat([float(i)], n1)
    y1 = x1 * x2

    root["x1"].append(x1)
    root["x2"].append(x2)
    root["y1"].append(y1)

duration_direct = time.perf_counter() - t_start
print(f"Strategy 1 (direct zarr append): {duration_direct:.3f}s")

Strategy 1 (direct zarr append): 11.136s


## Strategy 2: Pre-allocate with known final size + region writes

If we know the final size, we can pre-allocate the zarr arrays and write to specific regions. This avoids resize operations entirely.

In [3]:
fname2 = str(tmpdir / "prealloc.zarr")

total_size = n1 * n2

t_start = time.perf_counter()

# Pre-allocate zarr arrays with known final size
store = zarr.open_group(fname2, mode="w")
store.create_array("x1", shape=(total_size,), chunks=(10000,), dtype="float64")
store.create_array("x2", shape=(total_size,), chunks=(10000,), dtype="float64")
store.create_array("y1", shape=(total_size,), chunks=(10000,), dtype="float64")

# Write data in regions
for i in range(n2):
    x1 = np.arange(n1)
    x2 = np.repeat([float(i)], n1)
    y1 = x1 * x2

    slc = slice(i * n1, (i + 1) * n1)
    store["x1"][slc] = x1
    store["x2"][slc] = x2
    store["y1"][slc] = y1

duration_prealloc = time.perf_counter() - t_start
print(f"Strategy 2 (pre-allocate + region write): {duration_prealloc:.3f}s")

Strategy 2 (pre-allocate + region write): 7.304s


## Strategy 3: Direct zarr with unknown final size (resize as needed)

When final size is unknown, we can still use the direct zarr API but resize in chunk-aligned increments to minimize resize overhead.

In [4]:
fname3 = str(tmpdir / "resize.zarr")

chunk_size = 10000

t_start = time.perf_counter()

# Create zarr arrays with initial zero-size (will grow via append)
store = zarr.open_group(fname3, mode="w")
store.create_array("x1", shape=(0,), chunks=(chunk_size,), dtype="float64")
store.create_array("x2", shape=(0,), chunks=(chunk_size,), dtype="float64")
store.create_array("y1", shape=(0,), chunks=(chunk_size,), dtype="float64")

# Append data - zarr handles the resize internally
for i in range(n2):
    x1 = np.arange(n1)
    x2 = np.repeat([float(i)], n1)
    y1 = x1 * x2

    store["x1"].append(x1)
    store["x2"].append(x2)
    store["y1"].append(y1)

duration_resize = time.perf_counter() - t_start
print(f"Strategy 3 (zarr resize/append, unknown size): {duration_resize:.3f}s")

Strategy 3 (zarr resize/append, unknown size): 10.323s


## Strategy 4: Batched xarray writes (append every N iterations)

Trade-off: buffer multiple iterations in memory and write less frequently. Data in the buffer is at risk on crash, but disk I/O is reduced.

In [5]:
fname4 = str(tmpdir / "batched.zarr")

batch_size = 10  # Write every 10 iterations (10k points = 1 chunk)

t_start = time.perf_counter()

x1_buf, x2_buf, y1_buf = [], [], []

for i in range(n2):
    x1 = np.arange(n1)
    x2 = np.repeat([float(i)], n1)
    y1 = x1 * x2

    x1_buf.append(x1)
    x2_buf.append(x2)
    y1_buf.append(y1)

    if (i + 1) % batch_size == 0 or i == n2 - 1:
        ds = xr.Dataset(
            {"y1": (["t"], np.concatenate(y1_buf))},
            coords={
                "x1": ("t", np.concatenate(x1_buf)),
                "x2": ("t", np.concatenate(x2_buf)),
            },
        )
        if i < batch_size:
            ds.to_zarr(
                fname4,
                consolidated=False,
                encoding={
                    "x1": {"chunks": (10000,)},
                    "x2": {"chunks": (10000,)},
                    "y1": {"chunks": (10000,)},
                },
            )
        else:
            ds.to_zarr(fname4, append_dim="t", consolidated=False)
        x1_buf, x2_buf, y1_buf = [], [], []

duration_batched = time.perf_counter() - t_start
print(f"Strategy 4 (batched xarray, batch_size={batch_size}): {duration_batched:.3f}s")

Strategy 4 (batched xarray, batch_size=10): 1.181s


## Strategy 5: xarray `region` writes (pre-allocated, xarray-compatible)

Like Strategy 2 but using xarray's `region` parameter to write slices into a pre-allocated zarr store. Keeps xarray metadata intact.

In [6]:
fname5 = str(tmpdir / "region.zarr")

total_size = n1 * n2

t_start = time.perf_counter()

# Create pre-allocated store with xarray (preserves xarray metadata)
ds_template = xr.Dataset(
    {"y1": (["t"], np.zeros(total_size))},
    coords={
        "x1": ("t", np.zeros(total_size)),
        "x2": ("t", np.zeros(total_size)),
    },
)
ds_template.to_zarr(
    fname5,
    consolidated=False,
    compute=False,
    encoding={
        "x1": {"chunks": (10000,)},
        "x2": {"chunks": (10000,)},
        "y1": {"chunks": (10000,)},
    },
)

# Write regions
for i in range(n2):
    x1 = np.arange(n1)
    x2 = np.repeat([float(i)], n1)
    y1 = x1 * x2

    ds = xr.Dataset(
        {"y1": (["t"], y1)},
        coords={"x1": ("t", x1), "x2": ("t", x2)},
    )
    region = {"t": slice(i * n1, (i + 1) * n1)}
    ds.to_zarr(fname5, region=region, consolidated=False)

duration_region = time.perf_counter() - t_start
print(f"Strategy 5 (xarray region writes, pre-allocated): {duration_region:.3f}s")

Strategy 5 (xarray region writes, pre-allocated): 11.580s


### Investigating Strategy 5 overhead

Strategy 5 still creates an `xr.Dataset` and calls `to_zarr` each iteration. Let's break down where time is spent: dataset creation vs. the `to_zarr` region write.

In [7]:
fname5b = str(tmpdir / "region_profile.zarr")

total_size = n1 * n2

# Create pre-allocated store
ds_template = xr.Dataset(
    {"y1": (["t"], np.zeros(total_size))},
    coords={
        "x1": ("t", np.zeros(total_size)),
        "x2": ("t", np.zeros(total_size)),
    },
)
ds_template.to_zarr(
    fname5b,
    consolidated=False,
    compute=False,
    encoding={
        "x1": {"chunks": (10000,)},
        "x2": {"chunks": (10000,)},
        "y1": {"chunks": (10000,)},
    },
)

# Time the components separately
time_data_gen = 0.0
time_ds_create = 0.0
time_to_zarr = 0.0

for i in range(n2):
    t0 = time.perf_counter()
    x1 = np.arange(n1)
    x2 = np.repeat([float(i)], n1)
    y1 = x1 * x2
    t1 = time.perf_counter()

    ds = xr.Dataset(
        {"y1": (["t"], y1)},
        coords={"x1": ("t", x1), "x2": ("t", x2)},
    )
    t2 = time.perf_counter()

    region = {"t": slice(i * n1, (i + 1) * n1)}
    ds.to_zarr(fname5b, region=region, consolidated=False)
    t3 = time.perf_counter()

    time_data_gen += t1 - t0
    time_ds_create += t2 - t1
    time_to_zarr += t3 - t2

print(f"Data generation:    {time_data_gen:.3f}s")
print(f"xr.Dataset creation: {time_ds_create:.3f}s")
print(f"ds.to_zarr(region): {time_to_zarr:.3f}s")
print(f"Total:              {time_data_gen + time_ds_create + time_to_zarr:.3f}s")
print(
    f"\nto_zarr is {time_to_zarr / (time_data_gen + time_ds_create + time_to_zarr) * 100:.0f}% of total time"
)

Data generation:    0.012s
xr.Dataset creation: 0.060s
ds.to_zarr(region): 8.852s
Total:              8.925s

to_zarr is 99% of total time


### Strategy 5b: Pre-allocate with xarray, but write regions via direct zarr

This combines the best of both: xarray sets up the metadata-rich store, but subsequent writes go directly to zarr arrays (no per-iteration xarray overhead).

In [8]:
fname5c = str(tmpdir / "region_direct.zarr")

total_size = n1 * n2

t_start = time.perf_counter()

# Create pre-allocated store with xarray (preserves xarray metadata)
ds_template = xr.Dataset(
    {"y1": (["t"], np.zeros(total_size))},
    coords={
        "x1": ("t", np.zeros(total_size)),
        "x2": ("t", np.zeros(total_size)),
    },
)
ds_template.to_zarr(
    fname5c,
    consolidated=False,
    compute=False,
    encoding={
        "x1": {"chunks": (10000,)},
        "x2": {"chunks": (10000,)},
        "y1": {"chunks": (10000,)},
    },
)

# Open with zarr directly for fast region writes
root = zarr.open_group(fname5c, mode="r+")
for i in range(n2):
    x1 = np.arange(n1)
    x2 = np.repeat([float(i)], n1)
    y1 = x1 * x2

    slc = slice(i * n1, (i + 1) * n1)
    root["x1"][slc] = x1
    root["x2"][slc] = x2
    root["y1"][slc] = y1

duration_region_direct = time.perf_counter() - t_start
print(
    f"Strategy 5b (xarray pre-alloc + direct zarr region write): {duration_region_direct:.3f}s"
)
print(f"Speedup vs Strategy 5: {duration_region / duration_region_direct:.1f}x")

# Verify it's still readable as xarray
ds_check = xr.open_zarr(fname5c)
print(f"Readable as xarray: {ds_check.dims}")

Strategy 5b (xarray pre-alloc + direct zarr region write): 6.912s
Speedup vs Strategy 5: 1.7x
Readable as xarray: FrozenMappingWarningOnValuesAccess({'t': 200000})


C:\Users\jenielse\AppData\Local\Temp\ipykernel_2944\3186047673.py:45: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  ds_check = xr.open_zarr(fname5c)


### Analysis

The profiling shows `ds.to_zarr(region=...)` accounts for **99% of time** in Strategy 5. Each call takes ~57ms of overhead for just 1000 floats (8KB). This overhead comes from xarray re-opening the store, validating schemas, and checking coordinate alignment on every call.

Key findings from the summary table:
- **Strategy 2** (direct zarr region write): ~2x faster — skips xarray validation overhead
- **Strategy 5** (xarray region): barely faster than baseline — same per-call xarray overhead
- **Strategy 4** (batched): **~13x faster** — the real win is reducing call count from 200 → 20

The fundamental bottleneck is **per-call overhead × number of calls**, not raw I/O. Writing 1000 points per call means 200 calls, each with ~30-60ms overhead regardless of data size.

The optimal approach is: **batch writes to align with chunk boundaries + use direct zarr API**.

### Strategy 6: Best of both — xarray metadata + direct zarr + chunk-aligned batching

Pre-allocate with xarray (for metadata), buffer writes in memory, flush to zarr directly when a full chunk is ready. Crash-safe per chunk.

In [9]:
fname6 = str(tmpdir / "best_combined.zarr")

chunk_size = 10000
total_size = n1 * n2

t_start = time.perf_counter()

# 1. Pre-allocate with xarray (sets up coords, attrs, metadata)
ds_template = xr.Dataset(
    {"y1": (["t"], np.zeros(total_size))},
    coords={
        "x1": ("t", np.zeros(total_size)),
        "x2": ("t", np.zeros(total_size)),
    },
)
ds_template.to_zarr(
    fname6,
    consolidated=False,
    compute=False,
    encoding={
        "x1": {"chunks": (chunk_size,)},
        "x2": {"chunks": (chunk_size,)},
        "y1": {"chunks": (chunk_size,)},
    },
)

# 2. Open with zarr for direct writes, buffer to chunk boundaries
root = zarr.open_group(fname6, mode="r+")
write_offset = 0
x1_buf, x2_buf, y1_buf = [], [], []
buf_len = 0

for i in range(n2):
    x1 = np.arange(n1)
    x2 = np.repeat([float(i)], n1)
    y1 = x1 * x2

    x1_buf.append(x1)
    x2_buf.append(x2)
    y1_buf.append(y1)
    buf_len += n1

    # Flush when buffer fills a chunk
    if buf_len >= chunk_size:
        slc = slice(write_offset, write_offset + buf_len)
        root["x1"][slc] = np.concatenate(x1_buf)
        root["x2"][slc] = np.concatenate(x2_buf)
        root["y1"][slc] = np.concatenate(y1_buf)
        write_offset += buf_len
        x1_buf, x2_buf, y1_buf = [], [], []
        buf_len = 0

# Flush remaining
if buf_len > 0:
    slc = slice(write_offset, write_offset + buf_len)
    root["x1"][slc] = np.concatenate(x1_buf)
    root["x2"][slc] = np.concatenate(x2_buf)
    root["y1"][slc] = np.concatenate(y1_buf)

duration_best = time.perf_counter() - t_start
print(
    f"Strategy 6 (xarray metadata + zarr direct + chunk batching): {duration_best:.3f}s"
)
print(f"Speedup vs baseline: {duration / duration_best:.1f}x")

# Verify still readable as xarray
ds_check = xr.open_zarr(fname6, consolidated=False)
print(f"Readable as xarray: {ds_check.dims}")

Strategy 6 (xarray metadata + zarr direct + chunk batching): 0.575s
Speedup vs baseline: 24.8x
Readable as xarray: FrozenMappingWarningOnValuesAccess({'t': 200000})


### Strategy 7: Background thread writer

Offload disk I/O to a background thread. The main thread generates data and puts it on a queue; the writer thread flushes to zarr. This overlaps computation with I/O so the main loop never blocks on disk writes (except when the queue is full).

Two variants:
- **7a**: Write every iteration (per-iteration crash safety, maximum overlap)
- **7b**: Chunk-aligned batching in the writer thread (fewer writes, still async)

In [10]:
import queue
import threading

# --- Strategy 7a: Background thread, write per iteration ---
fname7a = str(tmpdir / "thread_per_iter.zarr")

total_size = n1 * n2
chunk_size = 10000

# Pre-allocate with xarray
ds_template = xr.Dataset(
    {"y1": (["t"], np.zeros(total_size))},
    coords={
        "x1": ("t", np.zeros(total_size)),
        "x2": ("t", np.zeros(total_size)),
    },
)
ds_template.to_zarr(
    fname7a,
    consolidated=False,
    compute=False,
    encoding={
        "x1": {"chunks": (chunk_size,)},
        "x2": {"chunks": (chunk_size,)},
        "y1": {"chunks": (chunk_size,)},
    },
)


def writer_thread_fn(zarr_path, q):
    """Background writer: reads (offset, x1, x2, y1) from queue until None sentinel."""
    root = zarr.open_group(zarr_path, mode="r+")
    while True:
        item = q.get()
        if item is None:
            break
        offset, x1_data, x2_data, y1_data = item
        slc = slice(offset, offset + len(x1_data))
        root["x1"][slc] = x1_data
        root["x2"][slc] = x2_data
        root["y1"][slc] = y1_data
        q.task_done()


t_start = time.perf_counter()

write_queue = queue.Queue(maxsize=20)  # Backpressure if writer falls behind
writer = threading.Thread(target=writer_thread_fn, args=(fname7a, write_queue))
writer.start()

for i in range(n2):
    x1 = np.arange(n1)
    x2 = np.repeat([float(i)], n1)
    y1 = x1 * x2
    # Enqueue write (blocks if queue full = backpressure)
    write_queue.put((i * n1, x1, x2, y1))

# Signal writer to stop and wait for completion
write_queue.put(None)
writer.join()

duration_thread = time.perf_counter() - t_start
print(f"Strategy 7a (background thread, per-iter writes): {duration_thread:.3f}s")
print(f"Speedup vs baseline: {duration / duration_thread:.1f}x")

Strategy 7a (background thread, per-iter writes): 6.769s
Speedup vs baseline: 2.1x


In [11]:
# --- Strategy 7b: Background thread + chunk-aligned batching ---
fname7b = str(tmpdir / "thread_chunked.zarr")

# Pre-allocate with xarray
ds_template = xr.Dataset(
    {"y1": (["t"], np.zeros(total_size))},
    coords={
        "x1": ("t", np.zeros(total_size)),
        "x2": ("t", np.zeros(total_size)),
    },
)
ds_template.to_zarr(
    fname7b,
    consolidated=False,
    compute=False,
    encoding={
        "x1": {"chunks": (chunk_size,)},
        "x2": {"chunks": (chunk_size,)},
        "y1": {"chunks": (chunk_size,)},
    },
)


def writer_chunked_thread_fn(zarr_path, q):
    """Background writer: receives pre-concatenated chunk-aligned buffers."""
    root = zarr.open_group(zarr_path, mode="r+")
    while True:
        item = q.get()
        if item is None:
            break
        offset, x1_data, x2_data, y1_data = item
        slc = slice(offset, offset + len(x1_data))
        root["x1"][slc] = x1_data
        root["x2"][slc] = x2_data
        root["y1"][slc] = y1_data
        q.task_done()


t_start = time.perf_counter()

write_queue = queue.Queue(maxsize=4)
writer = threading.Thread(target=writer_chunked_thread_fn, args=(fname7b, write_queue))
writer.start()

write_offset = 0
x1_buf, x2_buf, y1_buf = [], [], []
buf_len = 0

for i in range(n2):
    x1 = np.arange(n1)
    x2 = np.repeat([float(i)], n1)
    y1 = x1 * x2

    x1_buf.append(x1)
    x2_buf.append(x2)
    y1_buf.append(y1)
    buf_len += n1

    # When a full chunk is ready, send to writer thread
    if buf_len >= chunk_size:
        write_queue.put(
            (
                write_offset,
                np.concatenate(x1_buf),
                np.concatenate(x2_buf),
                np.concatenate(y1_buf),
            )
        )
        write_offset += buf_len
        x1_buf, x2_buf, y1_buf = [], [], []
        buf_len = 0

# Flush remaining
if buf_len > 0:
    write_queue.put(
        (
            write_offset,
            np.concatenate(x1_buf),
            np.concatenate(x2_buf),
            np.concatenate(y1_buf),
        )
    )

write_queue.put(None)
writer.join()

duration_thread_chunked = time.perf_counter() - t_start
print(
    f"Strategy 7b (background thread + chunk batching): {duration_thread_chunked:.3f}s"
)
print(f"Speedup vs baseline: {duration / duration_thread_chunked:.1f}x")
print(
    f"Speedup vs Strategy 6 (sync chunked): {duration_best / duration_thread_chunked:.2f}x"
)

Strategy 7b (background thread + chunk batching): 0.379s
Speedup vs baseline: 37.7x
Speedup vs Strategy 6 (sync chunked): 1.52x


## Summary comparison

## Theoretical maximum: single bulk write

Write the entire array at once (no incremental appending) to establish a lower bound on write time.

In [12]:
# Build the full arrays in memory first
x1_full = np.tile(np.arange(n1), n2)
x2_full = np.repeat(np.arange(n2, dtype="float64"), n1)
y1_full = x1_full * x2_full

# --- Bulk write via xarray ---
fname_bulk_xr = str(tmpdir / "bulk_xarray.zarr")

t_start = time.perf_counter()
ds_full = xr.Dataset(
    {"y1": (["t"], y1_full)},
    coords={"x1": ("t", x1_full), "x2": ("t", x2_full)},
)
ds_full.to_zarr(
    fname_bulk_xr,
    consolidated=False,
    encoding={
        "x1": {"chunks": (10000,)},
        "x2": {"chunks": (10000,)},
        "y1": {"chunks": (10000,)},
    },
)
duration_bulk_xr = time.perf_counter() - t_start
print(f"Bulk write (xarray):      {duration_bulk_xr:.3f}s")

# --- Bulk write via direct zarr ---
fname_bulk_zarr = str(tmpdir / "bulk_zarr.zarr")

t_start = time.perf_counter()
store = zarr.open_group(fname_bulk_zarr, mode="w")
store.create_array("x1", data=x1_full, chunks=(10000,))
store.create_array("x2", data=x2_full, chunks=(10000,))
store.create_array("y1", data=y1_full, chunks=(10000,))
duration_bulk_zarr = time.perf_counter() - t_start
print(f"Bulk write (direct zarr): {duration_bulk_zarr:.3f}s")

print(
    f"\nBaseline speedup: xarray bulk = {duration / duration_bulk_xr:.1f}x, zarr bulk = {duration / duration_bulk_zarr:.1f}x"
)

Bulk write (xarray):      0.183s
Bulk write (direct zarr): 0.162s

Baseline speedup: xarray bulk = 77.9x, zarr bulk = 88.4x


In [13]:
import pandas as pd

results = pd.DataFrame(
    {
        "Strategy": [
            "Baseline (xarray append_dim each iter)",
            "1: Direct zarr append",
            "2: Pre-allocate + region (zarr)",
            "3: Zarr resize/append (unknown size)",
            "4: Batched xarray (batch=10)",
            "5: xarray region writes (pre-alloc)",
            "5b: xarray pre-alloc + direct zarr region",
            "6: xarray meta + zarr direct + chunk batch",
            "7a: Background thread (per-iter)",
            "7b: Background thread + chunk batch",
            "Bulk write (xarray)",
            "Bulk write (direct zarr)",
        ],
        "Duration (s)": [
            duration,
            duration_direct,
            duration_prealloc,
            duration_resize,
            duration_batched,
            duration_region,
            duration_region_direct,
            duration_best,
            duration_thread,
            duration_thread_chunked,
            duration_bulk_xr,
            duration_bulk_zarr,
        ],
        "Crash-safe": [
            "per iteration",
            "per iteration",
            "per iteration",
            "per iteration",
            f"per {batch_size} iters",
            "per iteration",
            "per iteration",
            "per chunk",
            "per iteration",
            "per chunk",
            "none (all or nothing)",
            "none (all or nothing)",
        ],
        "Needs final size": [
            "No",
            "No",
            "Yes",
            "No",
            "No",
            "Yes",
            "Yes",
            "Yes",
            "Yes",
            "Yes",
            "Yes",
            "Yes",
        ],
    }
)
results["Speedup vs baseline"] = (
    results["Duration (s)"].iloc[0] / results["Duration (s)"]
)
results

,Strategy,Duration (s),Crash-safe,Needs final size,Speedup vs baseline
0,Baseline (xarray append_dim each iter),14.282089,per iteration,No,1.000000
1,1: Direct zarr append,11.136193,per iteration,No,1.282493
2,2: Pre-allocate + region (zarr),7.303693,per iteration,Yes,1.955461
3,3: Zarr resize/append (unknown size),10.322531,per iteration,No,1.383584
4,4: Batched xarray (batch=10),1.180525,per 10 iters,No,12.098087
5,5: xarray region writes (pre-alloc),11.579935,per iteration,Yes,1.233348
6,5b: xarray pre-alloc + direct zarr region,6.912469,per iteration,Yes,2.066134
7,6: xarray meta + zarr direct + chunk batch,0.575412,per chunk,Yes,24.820614
8,7a: Background thread (per-iter),6.769238,per iteration,Yes,2.109852
9,7b: Background thread + chunk batch,0.379090,per chunk,Yes,37.674685


In [14]:
# Cleanup entire temp directory
shutil.rmtree(tmpdir, ignore_errors=True)
print(f"Cleaned up: {tmpdir}")

Cleaned up: C:\Users\jenielse\AppData\Local\Temp\zarr_explore_qpzbwqny
